# RF-DETR Training Workflow

This notebook is a local wrapper for the RF-DETR training CLI. The CLI command is the source of truth; this notebook only configures paths, checks artifacts, and calls `face-benchmark train-rfdetr` when explicit run flags are enabled. See `../docs/rfdetr-training.md` for the full workflow.

## Dataset Policy

Training data and benchmark data must stay separate. Keep RF-DETR training datasets under `data/training/rfdetr/` and training outputs under `runs/training/`. Do not use `data/benchmark/target-video-test-3fps-clean/` for training, fine-tuning, augmentation experiments, or threshold tuning if it is being treated as test data. See `../docs/dataset-policy.md`.

In [ ]:
from pathlib import Path
import importlib.metadata
import importlib.util
import json
import subprocess
import sys


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "face_detection_benchmark").exists():
            return candidate
    raise RuntimeError("Could not find repo root from notebook location.")


def repo_path(path: Path) -> Path:
    return path if path.is_absolute() else REPO_ROOT / path


REPO_ROOT = find_repo_root(Path.cwd().resolve())
SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from face_detection_benchmark import config as benchmark_config

print(f"Repo root: {REPO_ROOT}")

In [ ]:
rfdetr_available = importlib.util.find_spec("rfdetr") is not None
pytorch_lightning_available = importlib.util.find_spec("pytorch_lightning") is not None

print(f"RF-DETR installed: {rfdetr_available}")
if rfdetr_available:
    print(f"RF-DETR version: {importlib.metadata.version('rfdetr')}")
print(f"PyTorch Lightning installed: {pytorch_lightning_available}")
if not pytorch_lightning_available:
    print("RF-DETR training dependencies may be missing. Install RF-DETR train extras before a real training run.")

In [ ]:
training_dataset_dir = repo_path(benchmark_config.DEFAULT_RFDETR_TRAINING_DATA_DIR)
training_runs_dir = repo_path(benchmark_config.DEFAULT_TRAINING_RUNS_DIR)
benchmark_data_dir = repo_path(benchmark_config.DEFAULT_BENCHMARK_DATA_DIR)

print(f"Training dataset dir exists: {training_dataset_dir.exists()} - {training_dataset_dir}")
print(f"Training runs dir exists: {training_runs_dir.exists()} - {training_runs_dir}")
print(f"Benchmark data dir: {benchmark_data_dir}")
print(f"Dataset path is under benchmark data: {benchmark_data_dir in training_dataset_dir.parents}")

## Run Training

Edit the variables in the next cell before enabling `RUN_TRAINING`. The dataset directory must be explicit and must not be inside `data/benchmark/`.

In [ ]:
RUN_TRAINING = False

TRAINING_RUN_ID = "rfdetr-training-smoke"
TRAINING_DATASET_DIR = repo_path(benchmark_config.DEFAULT_RFDETR_TRAINING_DATA_DIR)
TRAINING_OUTPUT_DIR = repo_path(benchmark_config.DEFAULT_TRAINING_RUNS_DIR) / TRAINING_RUN_ID
TRAINING_WEIGHTS_PATH = None
TRAINING_EPOCHS = 100
TRAINING_BATCH_SIZE = 4
TRAINING_DEVICE = "auto"
TRAINING_DATASET_FILE = "roboflow"
TRAINING_NUM_WORKERS = 2

print(f"Training dataset: {TRAINING_DATASET_DIR}")
print(f"Training output: {TRAINING_OUTPUT_DIR}")

In [ ]:
training_command = [
    "uv",
    "run",
    "face-benchmark",
    "train-rfdetr",
    "--dataset-dir",
    str(TRAINING_DATASET_DIR),
    "--output-dir",
    str(TRAINING_OUTPUT_DIR),
    "--epochs",
    str(TRAINING_EPOCHS),
    "--batch-size",
    str(TRAINING_BATCH_SIZE),
    "--device",
    TRAINING_DEVICE,
    "--dataset-file",
    TRAINING_DATASET_FILE,
    "--num-workers",
    str(TRAINING_NUM_WORKERS),
]
if TRAINING_WEIGHTS_PATH is not None:
    training_command.extend(["--weights", str(TRAINING_WEIGHTS_PATH)])

print(" ".join(training_command))

In [ ]:
if RUN_TRAINING:
    subprocess.run(training_command, cwd=REPO_ROOT, check=True)
else:
    print("Skipping RF-DETR training. Set RUN_TRAINING = True to run it.")

In [ ]:
config_path = TRAINING_OUTPUT_DIR / "config.json"
metadata_path = TRAINING_OUTPUT_DIR / "metadata.json"

print(f"Config exists: {config_path.exists()} - {config_path}")
print(f"Metadata exists: {metadata_path.exists()} - {metadata_path}")
if config_path.exists():
    display(json.loads(config_path.read_text(encoding="utf-8")))
if metadata_path.exists():
    display(json.loads(metadata_path.read_text(encoding="utf-8")))